# Structural Alerts & GNN Explainability — Tox21 SR-ARE

Two analyses in one notebook:
1. **Brenk/REOS structural alerts** — flag known reactive/toxic substructures in the dataset
2. **GNNExplainer + Integrated Gradients** — extract atom-level importance from the trained SAGE→GIN→SAGE model and check alignment with those alerts

Run on **Colab T4 GPU**. Mount Drive first.


## Mount Drive & set paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys

DRIVE_ROOT= '/content/drive/MyDrive/TOX' 
DATA_CSV = f'{DRIVE_ROOT}/tox21_dataset.csv'
CKPT_PATH = f'{DRIVE_ROOT}/checkpoints_tox21/sage_gin_sage_best/model_sage_gin_sage_run00_tox21.pt'
RESULTS_DIR = f'{DRIVE_ROOT}/results_tox21/explainability'
os.makedirs(RESULTS_DIR, exist_ok=True)
sys.path.insert(0, DRIVE_ROOT)

print('Drive mounted.')
print('Checkpoint path:', CKPT_PATH)
print('Results will go to:', RESULTS_DIR)


ModuleNotFoundError: No module named 'google.colab'

## Install dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_geometric', 'rdkit', 'captum'], check=True)
# captum provides Integrated Gradients
print('Done.')


## Imports

In [ ]:
import warnings, random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Linear

from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, GATConv, GINConv, SAGEConv
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp
from torch_geometric.explain import Explainer, GNNExplainer

from rdkit import Chem
from rdkit.Chem import Draw, AllChem, rdMolDescriptors
from rdkit.Chem.rdmolops import GetAdjacencyMatrix
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import display, Image
import io

from captum.attr import IntegratedGradients

from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## Load utils.py from Drive

In [ ]:
from utils import (
    get_atom_features,
    get_bond_features,
    create_pytorch_geometric_graph_data_list_from_smiles_and_labels as smiles_to_graphs,
    load_ckp,
    optimizer_to,
)
print('utils.py loaded.')


## GNN model definition
Must match the architecture of the saved checkpoint exactly.

In [ ]:
# These must match what was used when the checkpoint was saved
BEST_LAYER_TYPES = ['sage', 'gin', 'sage']
BEST_HIDDEN = 224
BEST_DROPOUT = 0.0006

class GNN(torch.nn.Module):
    def __init__(self, layer_types, hidden_dim, dropout):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        in_dim = 79
        for lt in layer_types:
            if lt == 'gcn':
                self.convs.append(GCNConv(in_dim, hidden_dim))
            elif lt == 'gat':
                self.convs.append(GATConv(in_dim, hidden_dim))
            elif lt == 'gin':
                mlp = nn.Sequential(
                    nn.Linear(in_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
                    nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
                self.convs.append(GINConv(mlp, eps=0.00005, train_eps=True))
            elif lt == 'sage':
                self.convs.append(SAGEConv(in_dim, hidden_dim))
            in_dim = hidden_dim
        self.drop = nn.Dropout(p=dropout)
        self.out  = Linear(hidden_dim * 2, 1)

    def forward(self, x, edge_index, batch_index):
        for conv in self.convs:
            x = conv(x, edge_index)
            x = torch.tanh(x)
        x = self.drop(x)
        x = torch.cat([gmp(x, batch_index), gap(x, batch_index)], dim=1)
        return self.out(x)

print('GNN defined.')


## Load trained checkpoint

In [ ]:
model = GNN(BEST_LAYER_TYPES, BEST_HIDDEN, BEST_DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0004)

if os.path.exists(CKPT_PATH):
    model, optimizer, start_epoch = load_ckp(CKPT_PATH, model, optimizer)
    print(f'Checkpoint loaded (trained to epoch {start_epoch})')
else:
    print('WARNING: checkpoint not found at', CKPT_PATH)
    print('The model will use random weights — explainability results will be meaningless.')
    print('Train the model first using the experiment notebook.')

model.eval()


## Load dataset

In [ ]:
df = pd.read_csv(DATA_CSV)
# df_task = df[['smiles', 'SR-ARE']].dropna().reset_index(drop=True)
print(f'Compounds: {len(df)}')

# Pre-filter invalid SMILES
X_smiles, y_labels = [], []

task_columns = df.columns[1:]  
for smi, y_val in zip(df['smiles'], df['SR-ARE']):
    if Chem.MolFromSmiles(smi) is not None:
        X_smiles.append(smi)
        y_labels.append(int(y_val))

print(f'Valid SMILES: {len(X_smiles)}')
print(f'Positive (toxic): {sum(y_labels)}  Negative: {len(y_labels)-sum(y_labels)}')


## Brenk & REOS Structural Alerts

These are curated lists of SMARTS patterns that flag known reactive or undesirable substructures.
- **Brenk alerts**: fragments associated with toxicity or metabolic instability
- **REOS filters**: Rapid Elimination Of Swill — flags generally problematic groups

We screen every molecule in the dataset and record which alerts fire.


In [ ]:
#  Brenk alert SMARTS (subset of the published list — most common reactive alerts) ──
# Full list: Brenk R. et al. ChemMedChem 2008
BRENK_ALERTS = {
    'Nitro_group':            '[N+](=O)[O-]',
    'Michael_acceptor':       'C=CC(=O)',
    'Aldehyde':               '[CX3H1](=O)',
    'Epoxide':                'C1OC1',
    'Acyl_halide':            'C(=O)[F,Cl,Br,I]',
    'Isocyanate':             'N=C=O',
    'Thiocyanate':            'SC#N',
    'Diazo':                  'N#N',
    'Peroxide':               'OO',
    'Halogenated_heterocycle':'c1ccnc(Cl)c1',
    'Heavy_metal':            '[As,Se,Hg,Pb,Cd,Tl]',
    'Alkyl_halide_reactive':  '[CX4][Cl,Br,I]',
    'Aniline':                'Nc1ccccc1',
    'Hydrazine':              'NN',
    'Hydroxamic_acid':        'C(=O)NO',
}

#  REOS filters (reactive functional groups) ──
REOS_ALERTS = {
    'Sulfonyl_halide':   'S(=O)(=O)[Cl,F]',
    'Acid_anhydride':    'C(=O)OC(=O)',
    'Activated_ester':   'C(=O)O[N,c]',
    'Aziridine':         'C1CN1',
    'Vinyl_halide':      'C=C[Cl,Br]',
    'Phosphonate':       'P(=O)(O)O',
    'Thiol':             '[SH]',
    'Disulfide':         'SS',
}

ALL_ALERTS = {**BRENK_ALERTS, **REOS_ALERTS}

# Compile SMARTS
compiled = {name: Chem.MolFromSmarts(sma) for name, sma in ALL_ALERTS.items()}

# Screen dataset
alert_records = []
for smi, y_val in zip(X_smiles, y_labels):
    mol = Chem.MolFromSmiles(smi)
    row = {'smiles': smi, 'toxic': y_val}
    fired = []
    for name, patt in compiled.items():
        if patt is not None and mol.HasSubstructMatch(patt):
            row[name] = 1
            fired.append(name)
        else:
            row[name] = 0
    row['n_alerts_fired'] = len(fired)
    row['any_alert'] = int(len(fired) > 0)
    alert_records.append(row)

alert_df = pd.DataFrame(alert_records)
alert_names = list(ALL_ALERTS.keys())

print(f'Molecules with at least one alert: {alert_df["any_alert"].sum()} '
      f'/ {len(alert_df)} ({100*alert_df["any_alert"].mean():.1f}%)')
print()
print('Alert frequency:')
freq = alert_df[alert_names].sum().sort_values(ascending=False)
print(freq[freq > 0].to_string())


## Alert presence vs toxicity label

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: alert frequency split by toxic/non-toxic ────────────────────────
toxic_df    = alert_df[alert_df['toxic'] == 1]
nontoxic_df = alert_df[alert_df['toxic'] == 0]

alert_toxic    = toxic_df[alert_names].mean().sort_values(ascending=False)
alert_nontoxic = nontoxic_df[alert_names].reindex(alert_toxic.index)

x = range(len(alert_names))
axes[0].bar([i - 0.2 for i in x], alert_toxic.values,    width=0.4, label='Toxic (SR-ARE=1)',    color='tomato',    alpha=0.8)
axes[0].bar([i + 0.2 for i in x], alert_nontoxic.values, width=0.4, label='Non-toxic (SR-ARE=0)', color='steelblue', alpha=0.8)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(alert_toxic.index, rotation=60, ha='right', fontsize=8)
axes[0].set_ylabel('Fraction of compounds with alert')
axes[0].set_title('Alert prevalence: toxic vs non-toxic')
axes[0].legend()

# ── Right: toxicity rate by number of alerts fired ────────────────────────
grouped = alert_df.groupby('n_alerts_fired')['toxic'].mean().reset_index()
axes[1].bar(grouped['n_alerts_fired'], grouped['toxic'], color='darkorange', alpha=0.8)
axes[1].set_xlabel('Number of alerts fired per molecule')
axes[1].set_ylabel('Fraction labelled toxic (SR-ARE=1)')
axes[1].set_title('More alerts → higher toxicity rate?')

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'structural_alerts_vs_toxicity.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Saved to {plot_path}')

# ── Save alert table ───────────────────────────────────────────────────────
csv_path = os.path.join(RESULTS_DIR, 'structural_alerts_per_molecule.csv')
alert_df.to_csv(csv_path, index=False)
print(f'Alert table saved to {csv_path}')


## GNNExplainer — atom importance masks

GNNExplainer learns a soft mask over edges (and optionally nodes) that maximally preserves
the model's prediction for a given molecule. High-mask atoms are the ones the model
"looks at" when predicting toxicity.

We run it on a sample of **toxic molecules** and check whether the top-importance atoms
belong to known structural alert substructures.


In [ ]:
from torch_geometric.explain import Explainer, GNNExplainer

# ── Build a GNNExplainer wrapper ──────────────────────────────────────────
explainer = Explainer(
    model=model,
    algorithm=GNNExplainer(epochs=200),
    explanation_type='model',
    node_mask_type='attributes',
    edge_mask_type='object',
    model_config=dict(
        mode='binary_classification',
        task_level='graph',
        return_type='raw',        # model returns logits
    ),
)

# ── Pick a sample of toxic molecules to explain ───────────────────────────
N_EXPLAIN = 10   # increase for more thorough analysis; each takes ~5s
toxic_smiles = [smi for smi, y in zip(X_smiles, y_labels) if y == 1][:N_EXPLAIN]

print(f'Running GNNExplainer on {N_EXPLAIN} toxic molecules...')


In [ ]:
def smiles_to_single_graph(smi):
    """Convert one SMILES to a PyG Data object (no label needed)."""
    mol = Chem.MolFromSmiles(smi)
    n   = mol.GetNumAtoms()
    ref = Chem.MolFromSmiles('O=O')
    nf  = len(get_atom_features(ref.GetAtomWithIdx(0)))
    ef  = len(get_bond_features(ref.GetBondBetweenAtoms(0, 1)))
    X   = np.zeros((n, nf))
    for atom in mol.GetAtoms():
        X[atom.GetIdx()] = get_atom_features(atom)
    X = torch.tensor(X, dtype=torch.float)
    rows, cols = np.nonzero(GetAdjacencyMatrix(mol))
    E  = torch.stack([torch.from_numpy(rows.astype(np.int64)),
                      torch.from_numpy(cols.astype(np.int64))], dim=0)
    ne = 2 * mol.GetNumBonds()
    EF = np.zeros((ne, ef))
    for k, (i, j) in enumerate(zip(rows, cols)):
        EF[k] = get_bond_features(mol.GetBondBetweenAtoms(int(i), int(j)))
    EF = torch.tensor(EF, dtype=torch.float)
    batch = torch.zeros(n, dtype=torch.long)
    return Data(x=X, edge_index=E, edge_attr=EF, batch=batch)


def get_atom_importance(explanation):
    """Sum node mask features per atom → single importance score per atom."""
    if explanation.node_mask is not None:
        return explanation.node_mask.sum(dim=1).cpu().numpy()
    return np.zeros(explanation.x.shape[0])


explanations = []
for smi in toxic_smiles:
    data = smiles_to_single_graph(smi).to(device)
    exp  = explainer(data.x.float(), data.edge_index, batch=data.batch)
    atom_imp = get_atom_importance(exp)
    explanations.append({'smiles': smi, 'atom_importance': atom_imp})
    print(f'  Done: {smi[:50]}...' if len(smi) > 50 else f'  Done: {smi}')

print(f'\nGNNExplainer finished for {len(explanations)} molecules.')


## 11 · Visualise atom importance on molecule structures

In [ ]:
def draw_mol_with_importance(smi, atom_importance, title='', size=(400, 300)):
    """Draw molecule with atoms coloured by GNNExplainer importance (blue=low, red=high)."""
    mol = Chem.MolFromSmiles(smi)
    mol = Chem.RWMol(mol)
    AllChem.Compute2DCoords(mol)

    imp = np.array(atom_importance)
    imp_norm = (imp - imp.min()) / (imp.max() - imp.min() + 1e-8)

    atom_colours = {}
    atom_radii   = {}
    for i, v in enumerate(imp_norm):
        r, g, b = cm.RdYlBu_r(v)[:3]   # red=high importance, blue=low
        atom_colours[i] = (r, g, b)
        atom_radii[i]   = 0.3 + 0.4 * v

    drawer = rdMolDraw2D.MolDraw2DSVG(size[0], size[1])
    drawer.drawOptions().addAtomIndices = False
    rdMolDraw2D.PrepareMolForDrawing(mol)
    drawer.DrawMolecule(mol,
                        highlightAtoms=list(range(mol.GetNumAtoms())),
                        highlightAtomColors=atom_colours,
                        highlightAtomRadii=atom_radii,
                        highlightBonds=[])
    drawer.DrawMoleculeWithHighlights(mol, title,
        atom_colours, {}, atom_radii, {})
    drawer.FinishDrawing()
    return drawer.GetDrawingText()


# Draw first 4 explanations in a grid
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for idx, (exp_data, ax) in enumerate(zip(explanations[:4], axes)):
    smi = exp_data['smiles']
    imp = exp_data['atom_importance']
    mol = Chem.MolFromSmiles(smi)
    AllChem.Compute2DCoords(mol)

    imp_norm = (imp - imp.min()) / (imp.max() - imp.min() + 1e-8)
    highlight_atoms  = [i for i, v in enumerate(imp_norm) if v > 0.5]
    highlight_colors = {i: (1.0, 0.3 + 0.4*(1-imp_norm[i]), 0.3) for i in highlight_atoms}

    img = Draw.MolToImage(mol,
                          size=(400, 300),
                          highlightAtoms=highlight_atoms,
                          highlightColor=(1.0, 0.4, 0.4))
    ax.imshow(img)
    ax.set_title(f'Mol {idx+1} | Top atoms highlighted (importance > 0.5)', fontsize=8)
    ax.axis('off')

plt.suptitle('GNNExplainer: high-importance atoms in toxic molecules\n(red = model focuses here)', fontsize=12)
plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'gnnexplainer_toxic_molecules.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Saved to {plot_path}')


## 12 · Alert alignment analysis

For each explained molecule, check whether the **top-importance atoms** (from GNNExplainer)
overlap with atoms that belong to known structural alert substructures.

A high overlap means the model has implicitly learned to focus on chemically meaningful
reactive groups — strong evidence of interpretability.


In [ ]:
def get_alert_atom_indices(smi):
    """Return a dict of {alert_name: set of atom indices} for all alerts that fire."""
    mol = Chem.MolFromSmiles(smi)
    fired = {}
    for name, patt in compiled.items():
        if patt is not None and mol.HasSubstructMatch(patt):
            matches = mol.GetSubstructMatches(patt)
            atom_idxs = set(idx for match in matches for idx in match)
            fired[name] = atom_idxs
    return fired


alignment_results = []
for exp_data in explanations:
    smi = exp_data['smiles']
    imp = exp_data['atom_importance']
    imp_norm = (imp - imp.min()) / (imp.max() - imp.min() + 1e-8)
    top_atoms = set(np.where(imp_norm > 0.5)[0])   # atoms with importance > 50th percentile

    alert_atoms = get_alert_atom_indices(smi)
    all_alert_atoms = set(idx for s in alert_atoms.values() for idx in s)

    if len(top_atoms) == 0 or len(all_alert_atoms) == 0:
        overlap = 0.0
    else:
        overlap = len(top_atoms & all_alert_atoms) / len(top_atoms | all_alert_atoms)

    alignment_results.append({
        'smiles':           smi,
        'alerts_fired':     list(alert_atoms.keys()),
        'n_top_atoms':      len(top_atoms),
        'n_alert_atoms':    len(all_alert_atoms),
        'overlap_jaccard':  round(overlap, 4),
    })

align_df = pd.DataFrame(alignment_results)
print('Alert alignment (Jaccard overlap between GNNExplainer top atoms and alert atoms):')
print(align_df[['alerts_fired', 'n_top_atoms', 'n_alert_atoms', 'overlap_jaccard']].to_string())
print(f'\nMean Jaccard overlap: {align_df["overlap_jaccard"].mean():.3f}')
print('(1.0 = perfect alignment, 0.0 = no overlap)')

align_df.to_csv(os.path.join(RESULTS_DIR, 'gnnexplainer_alert_alignment.csv'), index=False)


## Integrated Gradients (Captum)

Integrated Gradients is a gradient-based attribution method that attributes the model's
output to each **input feature** (atom feature dimensions) by integrating gradients along
a path from a baseline (zero features) to the actual input.

This gives a finer-grained view than GNNExplainer: instead of one importance score per atom,
you get one score per atom **per feature dimension** — so you can see which chemical
properties (aromaticity, hybridisation, etc.) drive the prediction.


In [ ]:
class GNNForIG(torch.nn.Module):
    """
    Wrapper that accepts x as a positional arg so Captum's IG can
    differentiate with respect to it. edge_index and batch are fixed
    via partial application.
    """
    def __init__(self, gnn, edge_index, batch):
        super().__init__()
        self.gnn        = gnn
        self.edge_index = edge_index
        self.batch      = batch

    def forward(self, x):
        return torch.sigmoid(self.gnn(x, self.edge_index, self.batch))


def run_integrated_gradients(model, data, target_class=0, n_steps=50):
    """
    Run Integrated Gradients on a single molecule graph.
    Returns per-atom attribution (summed over feature dims).
    """
    model.eval()
    x          = data.x.float().to(device).requires_grad_(True)
    edge_index = data.edge_index.to(device)
    batch      = data.batch.to(device)

    wrapper    = GNNForIG(model, edge_index, batch).to(device)
    ig         = IntegratedGradients(wrapper)
    baseline   = torch.zeros_like(x)

    attributions, delta = ig.attribute(
        x,
        baselines=baseline,
        target=None,
        n_steps=n_steps,
        return_convergence_delta=True,
    )
    # Sum attribution over feature dimensions → one score per atom
    atom_attr = attributions.abs().sum(dim=1).detach().cpu().numpy()
    return atom_attr, float(delta.mean().abs())


print('Running Integrated Gradients on same toxic molecules...')
ig_results = []
for exp_data in explanations:
    smi  = exp_data['smiles']
    data = smiles_to_single_graph(smi).to(device)
    attr, delta = run_integrated_gradients(model, data)
    ig_results.append({'smiles': smi, 'atom_attr': attr, 'convergence_delta': delta})
    print(f'  IG done | convergence δ={delta:.4f} | {smi[:40]}')

print('\nIntegrated Gradients complete.')
print('(convergence_delta should be close to 0 — measures IG approximation error)')


In [ ]:
# ── Compare GNNExplainer vs IG importance scores ──────────────────────────
fig, axes = plt.subplots(2, min(4, len(explanations)), figsize=(14, 6))

for idx in range(min(4, len(explanations))):
    smi  = explanations[idx]['smiles']
    mol  = Chem.MolFromSmiles(smi)
    n    = mol.GetNumAtoms()
    labels = [mol.GetAtomWithIdx(i).GetSymbol() for i in range(n)]

    gnn_imp = explanations[idx]['atom_importance']
    ig_imp  = ig_results[idx]['atom_attr']
    gnn_n   = (gnn_imp - gnn_imp.min()) / (gnn_imp.max() - gnn_imp.min() + 1e-8)
    ig_n    = (ig_imp  - ig_imp.min())  / (ig_imp.max()  - ig_imp.min()  + 1e-8)

    axes[0, idx].bar(range(n), gnn_n, color='steelblue', alpha=0.8)
    axes[0, idx].set_title(f'Mol {idx+1}\nGNNExplainer', fontsize=8)
    axes[0, idx].set_xticks(range(n))
    axes[0, idx].set_xticklabels(labels, fontsize=6)

    axes[1, idx].bar(range(n), ig_n, color='darkorange', alpha=0.8)
    axes[1, idx].set_title(f'Mol {idx+1}\nIntegrated Gradients', fontsize=8)
    axes[1, idx].set_xticks(range(n))
    axes[1, idx].set_xticklabels(labels, fontsize=6)

axes[0, 0].set_ylabel('Normalised importance')
axes[1, 0].set_ylabel('Normalised attribution')
plt.suptitle('GNNExplainer vs Integrated Gradients — atom importance per molecule', fontsize=11)
plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, 'gnnexplainer_vs_ig_comparison.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Saved to {plot_path}')


## 14 · Summary & interpretation

In [ ]:
print('='*60)
print('STRUCTURAL ALERTS SUMMARY')
print('='*60)
print(f'Total molecules screened   : {len(alert_df)}')
print(f'With at least one alert    : {alert_df["any_alert"].sum()} ({100*alert_df["any_alert"].mean():.1f}%)')
print(f'Toxic molecules            : {sum(y_labels)}')
print()
print('Top 5 alerts by frequency:')
print(freq.head(5).to_string())
print()
print('='*60)
print('GNNExplainer ALERT ALIGNMENT')
print('='*60)
print(f'Mean Jaccard overlap (GNNExplainer top atoms vs alert atoms):')
print(f'  {align_df["overlap_jaccard"].mean():.3f}')
print()
print('Interpretation:')
print('  > 0.5  → model consistently focuses on known reactive fragments (good)')
print('  0.2-0.5 → partial alignment — model uses alerts plus other features')
print('  < 0.2  → model uses features independent of known alerts')
print()
print('All outputs saved to:', RESULTS_DIR)
